# Preprocessing Pipeline
## Federated Learning for Cross-Hospital Disease Prediction
### Dataset: CDC BRFSS 2015 Diabetes Health Indicators (Binary, Full/Imbalanced)

Each preprocessing step is written as its own **standalone, reusable function**, defined in one cell and immediately called/tested in the next cell, so every stage can be inspected before moving on. At the end, all functions are chained together into a single `run_preprocessing_pipeline()` for reuse (e.g. per Federated Learning client).

**Design choices carried over from the EDA + strategy discussion:**
- Drop exact duplicates before splitting (prevents train/test leakage)
- Cap (winsorize) BMI at a clinical upper bound instead of deleting rows
- Keep binary/ordinal features as integer codes (no one-hot encoding needed — no nominal categories exist in this dataset)
- Standardize only the continuous features (BMI, MentHlth, PhysHlth); leave binary/ordinal features unscaled
- Produce **two feature sets**: `full` (all features) and `reduced` (drops GenHlth/PhysHlth/DiffWalk, the leakage-risk features) — the "two-track" comparison from the strategy discussion
- Stratified train/test split to preserve the ~86/14 class ratio
- Scaler is fit on the **training set only**, then applied to test — avoids leaking test-set statistics into training


## 1. Imports

Only three libraries are used: `pandas` for tabular data handling, `scikit-learn` for the scaler and the train/test split utility, and `os` for creating the output folder. Nothing else is needed for this pipeline.

In [1]:
import os
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import kagglehub
from kagglehub import KaggleDatasetAdapter


## 2. Load Data

**What it does:** Reads the raw CSV into a pandas DataFrame.

**Why it's a separate function:** Isolating I/O makes the pipeline reusable — e.g. later you can point this at a per-client CSV file in the Federated Learning simulation without touching any other function.

In [3]:
def load_data(filepath: str) -> pd.DataFrame:
  """
  Loads the raw dataset from a specified file within the Kaggle dataset.

  Parameters
  ----------
  filepath : str
      The name of the file to load from the Kaggle dataset.

  Returns
  -------
  pd.DataFrame
      The loaded dataset as a pandas DataFrame.
  """

  # Load the latest version from Kaggle Hub
  # The 'filepath' argument is the name of the file within the dataset
  df = kagglehub.load_dataset(
    KaggleDatasetAdapter.PANDAS,
    "alexteboul/diabetes-health-indicators-dataset",
    filepath,
    # Provide any additional arguments like
    # sql_query or pandas_kwargs. See the
    # documenation for more information:
    # https://github.com/Kaggle/kagglehub/blob/main/README.md#kaggledatasetadapterpandas
  )

  print("First 5 records:", df.head())
  print(f"Loaded data: {df.shape[0]} rows, {df.shape[1]} columns")
  return df

In [4]:
# Call: load the raw dataset
raw_df = load_data("diabetes_binary_health_indicators_BRFSS2015.csv")
raw_df.head()


/tmp/ipykernel_2569/3353326266.py:18: DeprecationWarning: Use dataset_load() instead of load_dataset(). load_dataset() will be removed in a future version.
  df = kagglehub.load_dataset(


Using Colab cache for faster access to the 'diabetes-health-indicators-dataset' dataset.
First 5 records:    Diabetes_binary  HighBP  HighChol  CholCheck   BMI  Smoker  Stroke  \
0              0.0     1.0       1.0        1.0  40.0     1.0     0.0   
1              0.0     0.0       0.0        0.0  25.0     1.0     0.0   
2              0.0     1.0       1.0        1.0  28.0     0.0     0.0   
3              0.0     1.0       0.0        1.0  27.0     0.0     0.0   
4              0.0     1.0       1.0        1.0  24.0     0.0     0.0   

   HeartDiseaseorAttack  PhysActivity  Fruits  ...  AnyHealthcare  \
0                   0.0           0.0     0.0  ...            1.0   
1                   0.0           1.0     0.0  ...            0.0   
2                   0.0           0.0     1.0  ...            1.0   
3                   0.0           1.0     1.0  ...            1.0   
4                   0.0           1.0     1.0  ...            1.0   

   NoDocbcCost  GenHlth  MentHlth  PhysH

,Diabetes_binary,HighBP,HighChol,CholCheck,BMI,Smoker,Stroke,HeartDiseaseorAttack,PhysActivity,Fruits,...,AnyHealthcare,NoDocbcCost,GenHlth,MentHlth,PhysHlth,DiffWalk,Sex,Age,Education,Income
0,0.0,1.0,1.0,1.0,40.0,1.0,0.0,0.0,0.0,0.0,...,1.0,0.0,5.0,18.0,15.0,1.0,0.0,9.0,4.0,3.0
1,0.0,0.0,0.0,0.0,25.0,1.0,0.0,0.0,1.0,0.0,...,0.0,1.0,3.0,0.0,0.0,0.0,0.0,7.0,6.0,1.0
2,0.0,1.0,1.0,1.0,28.0,0.0,0.0,0.0,0.0,1.0,...,1.0,1.0,5.0,30.0,30.0,1.0,0.0,9.0,4.0,8.0
3,0.0,1.0,0.0,1.0,27.0,0.0,0.0,0.0,1.0,1.0,...,1.0,0.0,2.0,0.0,0.0,0.0,0.0,11.0,3.0,6.0
4,0.0,1.0,1.0,1.0,24.0,0.0,0.0,0.0,1.0,1.0,...,1.0,0.0,2.0,3.0,0.0,0.0,0.0,11.0,5.0,4.0


## 3. Handle Missing Values

**What it does:** Checks for missing values and, if any are found, drops rows containing them (a safe default for a low-missing-rate dataset). If none are found (as confirmed in the EDA), the function simply reports that and returns the data unchanged.

**Why it's needed:** Even though the EDA confirmed zero missing values, this function is kept in the pipeline for **robustness and reusability** — if this pipeline is later reused on a different hospital's raw data (which may not be as clean), it will still handle missing values safely instead of silently crashing downstream.

In [5]:
def handle_missing_values(df: pd.DataFrame) -> pd.DataFrame:
    """
    Check for and handle missing values.

    Strategy: drop rows with any missing values. This is safe here because
    the EDA confirmed 0% missing values in the source data; the function
    is kept generic so the same pipeline is safe to reuse on other
    (potentially messier) hospital datasets in a Federated Learning setup.

    Parameters
    ----------
    df : pd.DataFrame

    Returns
    -------
    pd.DataFrame
        Dataset with no missing values.
    """
    n_missing = df.isnull().sum().sum()
    if n_missing == 0:
        print("No missing values found. Data unchanged.")
        return df

    print(f"Found {n_missing} missing values. Dropping affected rows.")
    df_clean = df.dropna().reset_index(drop=True)
    print(f"Rows before: {len(df)}, rows after: {len(df_clean)}")
    return df_clean


In [6]:
# Call: handle missing values
df_no_missing = handle_missing_values(raw_df)


No missing values found. Data unchanged.


## 4. Remove Duplicate Rows

**What it does:** Drops exact duplicate rows (identical values across every column).

**Why it's needed:** The EDA found ~9.5% exact duplicates. If duplicates end up split across the train and test sets later, the model would effectively be evaluated on rows it already memorized, inflating performance metrics. Removing them before the split guarantees this can't happen.

In [7]:
def remove_duplicates(df: pd.DataFrame) -> pd.DataFrame:
    """
    Remove exact duplicate rows from the dataset.

    Must be called BEFORE the train/test split, otherwise duplicate rows
    could end up on both sides of the split and silently leak information
    from train into test.

    Parameters
    ----------
    df : pd.DataFrame

    Returns
    -------
    pd.DataFrame
        Deduplicated dataset with a reset index.
    """
    n_before = len(df)
    df_dedup = df.drop_duplicates().reset_index(drop=True)
    n_after = len(df_dedup)
    print(f"Removed {n_before - n_after} duplicate rows "
          f"({(n_before - n_after) / n_before * 100:.2f}% of data)")
    return df_dedup


In [8]:
# Call: remove duplicate rows
df_dedup = remove_duplicates(df_no_missing)


Removed 24206 duplicate rows (9.54% of data)


## 5. Cap Outliers (Winsorize BMI)

**What it does:** Caps extreme BMI values at a clinically-justified upper bound (60) instead of deleting the rows.

**Why capping instead of deleting:** The EDA showed high-BMI respondents are exactly the population most relevant to diabetes prediction (severe obesity is a real, clinically meaningful risk category, not noise). Deleting those rows would throw away valuable minority-relevant signal. Capping limits the leverage of implausibly extreme values while keeping every other feature value in that row intact.

In [9]:
def cap_outliers(df: pd.DataFrame, column: str = "BMI", upper_bound: float = 60.0) -> pd.DataFrame:
    """
    Cap (winsorize) extreme values in a numeric column at a fixed upper bound.

    Chosen over deletion because extreme BMI values (e.g. severe obesity)
    are clinically meaningful and diabetes-relevant, not data errors -
    deleting these rows would remove exactly the patients the model most
    needs to learn to identify.

    Parameters
    ----------
    df : pd.DataFrame
    column : str
        Column to cap. Default is "BMI".
    upper_bound : float
        Maximum allowed value; anything above is capped to this value.

    Returns
    -------
    pd.DataFrame
        Dataset with the specified column capped.
    """
    df = df.copy()
    n_capped = (df[column] > upper_bound).sum()
    df[column] = df[column].clip(upper=upper_bound)
    print(f"Capped {n_capped} values in '{column}' at {upper_bound}")
    return df


In [10]:
# Call: cap extreme BMI values
df_capped = cap_outliers(df_dedup, column="BMI", upper_bound=60.0)


Capped 805 values in 'BMI' at 60.0


## 6. Encode Features

**What it does:** Ensures binary and ordinal columns are stored as proper integer types (they arrive as `float64` even though they represent discrete codes). No one-hot encoding is applied, because every categorical feature in this dataset is either binary or ordinal — there are no unordered ("nominal") categories that would need it.

**Why this matters:** Storing 0/1 flags and ordinal scales (Age, GenHlth, Education, Income) as raw floats can mislead automatic type-handling downstream and bloats memory unnecessarily. Casting to integer keeps the natural order of ordinal features intact, which one-hot encoding would destroy.

In [11]:
def encode_features(df: pd.DataFrame) -> pd.DataFrame:
    """
    Cast binary and ordinal columns to integer type.

    No one-hot encoding is used: every categorical feature in this dataset
    (binary flags and ordinal scales like Age/GenHlth/Education/Income) is
    already represented as an ordered numeric code, so casting to int
    preserves that order while one-hot encoding would discard it.

    Parameters
    ----------
    df : pd.DataFrame

    Returns
    -------
    pd.DataFrame
        Dataset with binary/ordinal columns cast to int, target excluded
        from casting logic but included in output unchanged.
    """
    df = df.copy()

    # Every column except BMI, MentHlth, PhysHlth is binary or ordinal
    continuous_cols = ["BMI", "MentHlth", "PhysHlth"]
    categorical_cols = [c for c in df.columns if c not in continuous_cols]

    for col in categorical_cols:
        df[col] = df[col].astype(int)

    print(f"Cast {len(categorical_cols)} binary/ordinal columns to int")
    print(f"Left {len(continuous_cols)} continuous columns as float: {continuous_cols}")
    return df


In [12]:
# Call: encode (cast) feature types
df_encoded = encode_features(df_capped)
df_encoded.dtypes


Cast 19 binary/ordinal columns to int
Left 3 continuous columns as float: ['BMI', 'MentHlth', 'PhysHlth']


,0
Diabetes_binary,int64
HighBP,int64
HighChol,int64
CholCheck,int64
BMI,float64
Smoker,int64
Stroke,int64
HeartDiseaseorAttack,int64
PhysActivity,int64
Fruits,int64


## 7. Feature Selection

**What it does:** Produces **two** versions of the feature set:
- **`full`** — every feature retained
- **`reduced`** — drops `GenHlth`, `PhysHlth`, and `DiffWalk`, the three features flagged in the EDA as likely reflecting *consequences* of diabetes (leakage risk) rather than independent risk factors

**Why two versions instead of one:** Rather than silently deleting the leakage-risk columns (which asserts they're a problem without proof) or silently keeping them (which risks inflated, clinically circular results), training and comparing both versions lets the leakage effect be *demonstrated* with real numbers in your report — a stronger academic contribution than either extreme.

In [13]:
def select_features(df: pd.DataFrame,
                     target_col: str = "Diabetes_binary",
                     leakage_risk_cols: list = None) -> dict:
    """
    Produce two feature-set variants: 'full' (all features) and 'reduced'
    (leakage-risk features removed), for a two-track comparison.

    Parameters
    ----------
    df : pd.DataFrame
    target_col : str
        Name of the target column.
    leakage_risk_cols : list of str, optional
        Columns considered possible data-leakage risks. Defaults to
        ["GenHlth", "PhysHlth", "DiffWalk"] based on the EDA findings.

    Returns
    -------
    dict
        {"full": df_full, "reduced": df_reduced}
    """
    if leakage_risk_cols is None:
        leakage_risk_cols = ["GenHlth", "PhysHlth", "DiffWalk"]

    df_full = df.copy()
    df_reduced = df.drop(columns=leakage_risk_cols).copy()

    print(f"Full feature set: {df_full.shape[1] - 1} features + target")
    print(f"Reduced feature set: {df_reduced.shape[1] - 1} features + target "
          f"(dropped: {leakage_risk_cols})")

    return {"full": df_full, "reduced": df_reduced}


In [14]:
# Call: build the full and reduced feature-set variants
feature_sets = select_features(df_encoded, target_col="Diabetes_binary")
df_full = feature_sets["full"]
df_reduced = feature_sets["reduced"]


Full feature set: 21 features + target
Reduced feature set: 18 features + target (dropped: ['GenHlth', 'PhysHlth', 'DiffWalk'])


## 8. Train-Test Split

**What it does:** Splits a dataset into train and test sets using a **stratified** split, meaning the ~86/14 class ratio found in the EDA is preserved in both the train and test sets.

**Why stratified, not plain random:** With this level of class imbalance, a plain random split could by chance produce a test set with a noticeably different class ratio than train, making evaluation results less reliable and harder to compare across runs.

In [15]:
def split_data(df: pd.DataFrame,
               target_col: str = "Diabetes_binary",
               test_size: float = 0.2,
               random_state: int = 42):
    """
    Perform a stratified train/test split.

    Parameters
    ----------
    df : pd.DataFrame
    target_col : str
    test_size : float
        Proportion of data to allocate to the test set.
    random_state : int
        Seed for reproducibility.

    Returns
    -------
    tuple
        (X_train, X_test, y_train, y_test)
    """
    X = df.drop(columns=[target_col])
    y = df[target_col]

    X_train, X_test, y_train, y_test = train_test_split(
        X, y,
        test_size=test_size,
        stratify=y,              # preserves class ratio in both splits
        random_state=random_state
    )

    print(f"Train set: {X_train.shape[0]} rows | Test set: {X_test.shape[0]} rows")
    print("Train class ratio:")
    print(y_train.value_counts(normalize=True))
    print("Test class ratio:")
    print(y_test.value_counts(normalize=True))

    return X_train, X_test, y_train, y_test


In [16]:
# Call: stratified split on the FULL feature set
X_train_full, X_test_full, y_train_full, y_test_full = split_data(
    df_full, target_col="Diabetes_binary", test_size=0.2, random_state=42
)


Train set: 183579 rows | Test set: 45895 rows
Train class ratio:
Diabetes_binary
0    0.847052
1    0.152948
Name: proportion, dtype: float64
Test class ratio:
Diabetes_binary
0    0.847064
1    0.152936
Name: proportion, dtype: float64


In [17]:
# Call: stratified split on the REDUCED feature set
X_train_red, X_test_red, y_train_red, y_test_red = split_data(
    df_reduced, target_col="Diabetes_binary", test_size=0.2, random_state=42
)


Train set: 183579 rows | Test set: 45895 rows
Train class ratio:
Diabetes_binary
0    0.847052
1    0.152948
Name: proportion, dtype: float64
Test class ratio:
Diabetes_binary
0    0.847064
1    0.152936
Name: proportion, dtype: float64


## 9. Feature Scaling

**What it does:** Standardizes (Z-score scales) only the continuous columns present in the data (`BMI`, and `MentHlth`/`PhysHlth` where present) — binary and ordinal columns are left untouched, since scaling a 0/1 flag or an already-ordered integer code adds no value and would only hurt interpretability.

**Why the scaler is fit on training data only:** Fitting the `StandardScaler` on the full dataset (before splitting) would let statistics from the test set (its mean/std) influence the transformation applied to the training set — a subtle form of data leakage. Fitting only on `X_train`, then applying that *same* fitted transformation to `X_test`, keeps the test set genuinely unseen.

In [18]:
def scale_features(X_train: pd.DataFrame, X_test: pd.DataFrame,
                    continuous_cols: list = None):
    """
    Standardize continuous features. The scaler is fit on X_train only,
    then applied to both X_train and X_test, to avoid leaking test-set
    statistics into the training process.

    Parameters
    ----------
    X_train : pd.DataFrame
    X_test : pd.DataFrame
    continuous_cols : list of str, optional
        Columns to scale. Defaults to ["BMI", "MentHlth", "PhysHlth"],
        restricted to whichever of these are actually present in X_train
        (the reduced feature set drops PhysHlth).

    Returns
    -------
    tuple
        (X_train_scaled, X_test_scaled, fitted_scaler)
    """
    if continuous_cols is None:
        continuous_cols = ["BMI", "MentHlth", "PhysHlth"]

    # Only scale columns that actually exist in this feature set
    continuous_cols = [c for c in continuous_cols if c in X_train.columns]

    X_train_scaled = X_train.copy()
    X_test_scaled = X_test.copy()

    scaler = StandardScaler()
    X_train_scaled[continuous_cols] = scaler.fit_transform(X_train[continuous_cols])
    X_test_scaled[continuous_cols] = scaler.transform(X_test[continuous_cols])

    print(f"Scaled columns: {continuous_cols}")
    print("(scaler fit on X_train only, then applied to X_test)")

    return X_train_scaled, X_test_scaled, scaler


In [19]:
# Call: scale the FULL feature set (fit on train, apply to test)
X_train_full_scaled, X_test_full_scaled, scaler_full = scale_features(
    X_train_full, X_test_full
)


Scaled columns: ['BMI', 'MentHlth', 'PhysHlth']
(scaler fit on X_train only, then applied to X_test)


In [20]:
# Call: scale the REDUCED feature set (fit on train, apply to test)
X_train_red_scaled, X_test_red_scaled, scaler_red = scale_features(
    X_train_red, X_test_red
)


Scaled columns: ['BMI', 'MentHlth']
(scaler fit on X_train only, then applied to X_test)


## 10. Save Processed Datasets

**What it does:** Writes the processed train/test splits (features + target recombined into single files for convenience) to an output folder as CSV files.

**Why save train/test separately rather than one processed file:** Keeping the split persistent on disk guarantees that every later modeling step (baseline models, federated client simulation, etc.) uses the *exact same* train/test partition — re-running a random split later could silently produce a different partition and make results non-reproducible.

In [21]:
def save_datasets(X_train, X_test, y_train, y_test,
                   output_dir: str, prefix: str, target_col: str = "Diabetes_binary"):
    """
    Save train/test features and target to CSV files in output_dir.

    Parameters
    ----------
    X_train, X_test, y_train, y_test : pd.DataFrame / pd.Series
    output_dir : str
        Folder to save files into (created if it doesn't exist).
    prefix : str
        Prefix for filenames, e.g. "full" or "reduced".
    target_col : str
        Name to use for the target column when recombining.

    Returns
    -------
    None
    """
    os.makedirs(output_dir, exist_ok=True)

    train_df = X_train.copy()
    train_df[target_col] = y_train.values
    test_df = X_test.copy()
    test_df[target_col] = y_test.values

    train_path = os.path.join(output_dir, f"{prefix}_train.csv")
    test_path = os.path.join(output_dir, f"{prefix}_test.csv")

    train_df.to_csv(train_path, index=False)
    test_df.to_csv(test_path, index=False)

    print(f"Saved: {train_path} ({train_df.shape[0]} rows)")
    print(f"Saved: {test_path} ({test_df.shape[0]} rows)")


In [22]:
# Call: save the FULL feature-set train/test files
save_datasets(
    X_train_full_scaled, X_test_full_scaled, y_train_full, y_test_full,
    output_dir="processed_data", prefix="full"
)


Saved: processed_data/full_train.csv (183579 rows)
Saved: processed_data/full_test.csv (45895 rows)


In [23]:
# Call: save the REDUCED feature-set train/test files
save_datasets(
    X_train_red_scaled, X_test_red_scaled, y_train_red, y_test_red,
    output_dir="processed_data", prefix="reduced"
)


Saved: processed_data/reduced_train.csv (183579 rows)
Saved: processed_data/reduced_test.csv (45895 rows)


## 11. Full Reusable Pipeline

**What it does:** Chains every function above into a single entry point, `run_preprocessing_pipeline()`, that takes a raw CSV path and produces saved, processed train/test files for both the full and reduced feature sets.

**Why this matters for the Federated Learning project specifically:** This function is written so it can be called once per simulated hospital client (each with its own raw CSV partition) and will apply *identical* preprocessing logic to every client — which is essential in FL, since inconsistent preprocessing across clients would make their local model updates incompatible when aggregated into the global model. The only step requiring care in a real multi-client setup is scaling: here the scaler is fit locally per call, but in a true FL simulation you would instead fit the scaler globally (or aggregate per-client statistics) before distributing it to clients, so every client scales features the same way.

In [24]:
def run_preprocessing_pipeline(filepath: str, output_dir: str = "processed_data") -> dict:
    """
    Run the complete preprocessing pipeline end-to-end on a raw CSV file.

    Steps: load -> handle missing values -> remove duplicates -> cap
    outliers -> encode -> feature selection (full/reduced) -> train/test
    split -> scale -> save.

    Parameters
    ----------
    filepath : str
        Path to the raw CSV file.
    output_dir : str
        Folder where processed CSVs will be saved.

    Returns
    -------
    dict
        All intermediate and final objects, keyed for easy inspection:
        {
            "df_full": ..., "df_reduced": ...,
            "X_train_full": ..., "X_test_full": ..., "y_train_full": ..., "y_test_full": ...,
            "X_train_reduced": ..., "X_test_reduced": ..., "y_train_reduced": ..., "y_test_reduced": ...,
            "scaler_full": ..., "scaler_reduced": ...
        }
    """
    print("=" * 60)
    print("STEP 1: Load data")
    df = load_data(filepath)

    print("\n" + "=" * 60)
    print("STEP 2: Handle missing values")
    df = handle_missing_values(df)

    print("\n" + "=" * 60)
    print("STEP 3: Remove duplicates")
    df = remove_duplicates(df)

    print("\n" + "=" * 60)
    print("STEP 4: Cap outliers (BMI)")
    df = cap_outliers(df, column="BMI", upper_bound=60.0)

    print("\n" + "=" * 60)
    print("STEP 5: Encode features")
    df = encode_features(df)

    print("\n" + "=" * 60)
    print("STEP 6: Feature selection (full + reduced)")
    feature_sets = select_features(df, target_col="Diabetes_binary")
    df_full, df_reduced = feature_sets["full"], feature_sets["reduced"]

    print("\n" + "=" * 60)
    print("STEP 7: Train/test split (full)")
    X_train_full, X_test_full, y_train_full, y_test_full = split_data(df_full)

    print("\n" + "=" * 60)
    print("STEP 7b: Train/test split (reduced)")
    X_train_red, X_test_red, y_train_red, y_test_red = split_data(df_reduced)

    print("\n" + "=" * 60)
    print("STEP 8: Scale features (full)")
    X_train_full, X_test_full, scaler_full = scale_features(X_train_full, X_test_full)

    print("\n" + "=" * 60)
    print("STEP 8b: Scale features (reduced)")
    X_train_red, X_test_red, scaler_red = scale_features(X_train_red, X_test_red)

    print("\n" + "=" * 60)
    print("STEP 9: Save processed datasets")
    save_datasets(X_train_full, X_test_full, y_train_full, y_test_full,
                  output_dir=output_dir, prefix="full")
    save_datasets(X_train_red, X_test_red, y_train_red, y_test_red,
                  output_dir=output_dir, prefix="reduced")

    print("\n" + "=" * 60)
    print("Pipeline complete.")

    return {
        "df_full": df_full, "df_reduced": df_reduced,
        "X_train_full": X_train_full, "X_test_full": X_test_full,
        "y_train_full": y_train_full, "y_test_full": y_test_full,
        "X_train_reduced": X_train_red, "X_test_reduced": X_test_red,
        "y_train_reduced": y_train_red, "y_test_reduced": y_test_red,
        "scaler_full": scaler_full, "scaler_reduced": scaler_red,
    }


In [25]:
# Call: run the entire pipeline end-to-end, from raw CSV to saved processed files
results = run_preprocessing_pipeline(
    "diabetes_binary_health_indicators_BRFSS2015.csv",
    output_dir="processed_data"
)


STEP 1: Load data


/tmp/ipykernel_2569/3353326266.py:18: DeprecationWarning: Use dataset_load() instead of load_dataset(). load_dataset() will be removed in a future version.
  df = kagglehub.load_dataset(


Using Colab cache for faster access to the 'diabetes-health-indicators-dataset' dataset.
First 5 records:    Diabetes_binary  HighBP  HighChol  CholCheck   BMI  Smoker  Stroke  \
0              0.0     1.0       1.0        1.0  40.0     1.0     0.0   
1              0.0     0.0       0.0        0.0  25.0     1.0     0.0   
2              0.0     1.0       1.0        1.0  28.0     0.0     0.0   
3              0.0     1.0       0.0        1.0  27.0     0.0     0.0   
4              0.0     1.0       1.0        1.0  24.0     0.0     0.0   

   HeartDiseaseorAttack  PhysActivity  Fruits  ...  AnyHealthcare  \
0                   0.0           0.0     0.0  ...            1.0   
1                   0.0           1.0     0.0  ...            0.0   
2                   0.0           0.0     1.0  ...            1.0   
3                   0.0           1.0     1.0  ...            1.0   
4                   0.0           1.0     1.0  ...            1.0   

   NoDocbcCost  GenHlth  MentHlth  PhysH

## 12. Verify Output

Quick sanity check confirming the processed files were written and look correct — this is not a new preprocessing step, just a final check on the pipeline's output.

In [26]:
for f in sorted(os.listdir("processed_data")):
    path = os.path.join("processed_data", f)
    check_df = pd.read_csv(path)
    print(f"{f}: {check_df.shape[0]} rows, {check_df.shape[1]} columns")


full_test.csv: 45895 rows, 22 columns
full_train.csv: 183579 rows, 22 columns
reduced_test.csv: 45895 rows, 19 columns
reduced_train.csv: 183579 rows, 19 columns


In [27]:
# Preview one of the processed files
pd.read_csv("processed_data/full_train.csv").head()


,HighBP,HighChol,CholCheck,BMI,Smoker,Stroke,HeartDiseaseorAttack,PhysActivity,Fruits,Veggies,...,NoDocbcCost,GenHlth,MentHlth,PhysHlth,DiffWalk,Sex,Age,Education,Income,Diabetes_binary
0,1,0,1,-0.408442,1,0,0,1,0,1,...,0,2,-0.455262,-0.517047,0,1,7,5,7,0
1,1,1,1,0.838066,0,0,0,1,0,1,...,0,3,-0.455262,-0.517047,0,0,10,4,3,1
2,0,1,1,-0.720069,0,0,0,1,1,1,...,0,2,-0.455262,-0.517047,0,0,10,6,8,1
3,0,1,1,-1.031696,0,0,0,1,1,1,...,0,2,-0.455262,-0.517047,0,0,13,5,5,0
4,1,0,1,0.214812,0,0,0,1,1,1,...,0,4,-0.455262,0.698722,1,0,7,4,5,0
